# Calcium imaging

---

In [ ]:
import os
import sys
import cv2
import argparse
import mrcfile
import nd2
import copy
import numpy as np
import pandas as pd
import tifffile as tif
import matplotlib.pyplot as plt
import matplotlib.patches as patches
%matplotlib inline

import scipy.stats as stats
from tqdm import tqdm
from PIL import Image
from scipy.stats import sem
from scipy.stats import mannwhitneyu
from scipy.stats import ranksums

---

In [ ]:
ASH_Adult_comp = []

In [ ]:
ASH_Dauer_comp = []

In [ ]:
AVA_Adult_comp = []

In [ ]:
AVA_Dauer_comp = []

In [ ]:
AVB_Adult_comp = []

In [ ]:
AVB_Dauer_comp = []

In [ ]:
AVD_Adult_comp = []

In [ ]:
AVD_Dauer_comp = []

In [ ]:
PVC_Adult_comp = []

In [ ]:
PVC_Dauer_comp = []

In [ ]:
ASH_AVD_ASH_Adult_comp = []

In [ ]:
ASH_AVD_ASH_Adult_ctrl = []

In [ ]:
ASH_AVD_AVD_Adult_comp = []

In [ ]:
ASH_AVD_AVD_Adult_ctrl = []

In [ ]:
AVB_AVD_AVB_Adult_comp = []

In [ ]:
AVB_AVD_AVB_Adult_ctrl = []

In [ ]:
AVB_AVD_AVD_Adult_comp = []

In [ ]:
AVB_AVD_AVD_Adult_ctrl = []

---

In [ ]:
# read nd2 files

calcium_data = nd2.imread(r'your_directory\your_file.nd2')  # change
print(calcium_data.shape)

In [ ]:
frame_example = calcium_data[50,:,:]

fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(6,6))
ax1.imshow(frame_example, cmap="bone")
plt.show()

print(frame_example.shape)


In [ ]:
# decide ROI

fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(6,6))

ax1.imshow(frame_example, cmap="bone")
rect = patches.Rectangle((700, 1200), 300, 300, linewidth=1, edgecolor='r', facecolor='none')
        # top left corner(ax1,ax0); (w,h) order
ax1.add_patch(rect)

plt.show()


In [ ]:
# plot ROI

ROI_example = calcium_data[50,1150:1600,650:1100]

fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(4,4))
ax1.imshow(ROI_example, cmap="bone")  # bone, seismic, gray
plt.axis('off')
plt.show()

print(ROI_example.shape)


In [ ]:
# apply threshold and check all frames

THRESH = 650  # change

for ii in range(calcium_data.shape[0]):
    Frame = np.copy(calcium_data[ii,1200:1500,700:1000])  # change ROI
    
    Frame[Frame < THRESH] = 0
    
    plt.figure()
    plt.imshow(Frame)
    plt.show()


In [ ]:
# get signal from ROI
    # threshed version -> get mean of non-zero values

S_raw = []

for ii in range(calcium_data.shape[0]):
    ROI = np.copy(calcium_data[ii,1200:1500,700:1000])  # change ROI
    
    ROI[ROI < THRESH] = 0
    S_raw.append(np.mean(np.sort(ROI[ROI>0], axis=None)[::-1]))

S_raw = np.array(S_raw); print(S_raw.shape)


In [ ]:
# get ΔF/F value
    # ΔF/F = [ F(t)-F(0) / F(0) ]

S_raw_dVV = np.zeros(S_raw.shape)


BA = np.mean(S_raw[:50])  # baseline activity

for ii in range(S_raw_dVV.shape[0]):
    S_raw_dVV[ii] = (S_raw[ii]-BA) / BA  # ΔF/F


In [ ]:
# plot

fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(8,4))

xpos = np.arange(1,len(S_raw)+1,1)
ax1.plot(xpos, S_raw_dVV, color="midnightblue", alpha=0.7,\
         linestyle='solid',linewidth=2.0, label="test")

ax1.vlines(50, np.min(S_raw_dVV), np.max(S_raw_dVV), color='gray', alpha=0.5, linestyle='dashed', linewidth=1.5)  # stimulus start
ax1.vlines(100, np.min(S_raw_dVV), np.max(S_raw_dVV), color='gray', alpha=0.5, linestyle='dashed', linewidth=1.5)  # stimulus end
ax1.axvspan(50, 100, alpha=0.3, color='tab:red')

ax1.set_title("Activity", y=1.01, fontsize=21, fontname='Arial')
ax1.set_xlabel("Frames", fontsize=18, fontname='Arial')
ax1.set_ylabel("ΔF/F", fontsize=18, fontname='Arial')

plt.show()

In [ ]:
ASH_Adult_comp.append(S_raw_dVV)  # CHECK list name

In [ ]:
# list -> numpy array :: CHECK list name

ASH_Adult_comp_np = np.asarray(ASH_Adult_comp); print(ASH_Adult_comp_np.shape)

## Plot

In [ ]:
# get mean, std, sem

from scipy.stats import sem

# mean

ASH_Adult_comp_mean = np.nanmean(ASH_Adult_comp_np, axis=0)
ASH_Dauer_comp_mean = np.nanmean(ASH_Dauer_comp_np, axis=0)
print(ASH_Adult_comp_mean.shape)


# standard deviation

ASH_Adult_comp_std = np.nanstd(ASH_Adult_comp_np, axis=0)
ASH_Dauer_comp_std = np.nanstd(ASH_Dauer_comp_np, axis=0)
print(ASH_Adult_comp_std.shape)


# standard error mean

ASH_Adult_comp_sem = sem(ASH_Adult_comp_np, nan_policy='omit', axis=0)
ASH_Dauer_comp_sem = sem(ASH_Dauer_comp_np, nan_policy='omit', axis=0)
print(ASH_Adult_comp_sem.shape)


In [ ]:
print(ASH_Adult_comp_np.shape)
print(ASH_Dauer_comp_np.shape)

In [ ]:
# plot - ASH


fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,4))

xpos = np.arange(1,201,1)
ax1.set_title("ASH", y=1.01, fontsize=18, fontname='sans-serif')
ax1.set_xlabel("Time (s)", fontsize=14, fontname='sans-serif')
ax1.set_ylabel("ΔF/F", fontsize=14, fontname='sans-serif')

ax1.plot(xpos, ASH_Adult_comp_mean, color='#1F77B4', alpha=0.7, linestyle='dashed', linewidth=2.0, label="Adult")
ax1.plot(xpos, ASH_Dauer_comp_mean, color='#FF7F0E', alpha=0.7, linestyle='dashed', linewidth=2.0, label="Dauer")

ax1.fill_between(xpos, ASH_Adult_comp_mean-ASH_Adult_comp_sem/2, ASH_Adult_comp_mean+ASH_Adult_comp_sem/2,\
                 alpha=0.3, edgecolor=None, facecolor='#1F77B4', linewidth=0, linestyle='dashdot')
ax1.fill_between(xpos, ASH_Dauer_comp_mean-ASH_Dauer_comp_sem/2, ASH_Dauer_comp_mean+ASH_Dauer_comp_sem/2,\
                 alpha=0.3, edgecolor=None, facecolor='#FF7F0E', linewidth=0, linestyle='dashdot')

ax1.axvspan(50, 100, alpha=0.15, color='tab:red')

ax1.legend(bbox_to_anchor=(1, 1.01), loc='upper left', fontsize=14)
ax1.set_xlim(-5,205)
ax1.set_ylim(-0.068,0.135)
ax1.set_xticks(np.arange(0,225,25))
ax1.set_xticklabels(["0","2.5","5.0","7.5","10.0","12.5","15.0","17.5","20.0"],
                    fontsize=11.5, fontname='sans-serif', rotation=0)

plt.show()

In [ ]:
# plot - AVA :: manuscript ver.

    # svg ; change color ; change font


from matplotlib.ticker import FormatStrFormatter


fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,4))

xpos = np.arange(1,201,1)
ax1.set_title("AVA", y=1.01, fontsize=18, fontname='sans-serif')
ax1.set_xlabel("Time (s)", fontsize=14, fontname='sans-serif')
ax1.set_ylabel("ΔF/F", fontsize=14, fontname='sans-serif')

ax1.plot(xpos, AVA_Adult_comp_mean, color='#1F77B4', alpha=0.7, linestyle='dashed', linewidth=2.0, label="Adult")
ax1.plot(xpos, AVA_Dauer_comp_mean, color='#FF7F0E', alpha=0.7, linestyle='dashed', linewidth=2.0, label="Dauer")

ax1.fill_between(xpos, AVA_Adult_comp_mean-AVA_Adult_comp_sem/2, AVA_Adult_comp_mean+AVA_Adult_comp_sem/2,\
                 alpha=0.3, edgecolor=None, facecolor='#1F77B4', linewidth=0, linestyle='dashdot')
ax1.fill_between(xpos, AVA_Dauer_comp_mean-AVA_Dauer_comp_sem/2, AVA_Dauer_comp_mean+AVA_Dauer_comp_sem/2,\
                 alpha=0.3, edgecolor=None, facecolor='#FF7F0E', linewidth=0, linestyle='dashdot')

ax1.axvspan(50, 100, alpha=0.15, color='tab:red')

ax1.legend(bbox_to_anchor=(1, 1.01), loc='upper left', fontsize=14)
ax1.set_xlim(-5,205)
ax1.set_ylim(-0.012,0.065)
ax1.set_xticks(np.arange(0,225,25))
ax1.set_xticklabels(["0","2.5","5.0","7.5","10.0","12.5","15.0","17.5","20.0"], fontsize=11.5, fontname='sans-serif', rotation=0)

# plt.savefig(r'D:\AVA_signal_comp_both_260202.svg', dpi=300, facecolor='w', bbox_inches='tight')
plt.show()


In [ ]:
# plot - AVB :: manuscript ver.

    # svg ; change color ; change font


from matplotlib.ticker import FormatStrFormatter


fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,4))

xpos1 = np.arange(1,201,1)
xpos2 = np.arange(1,201,1)
ax1.set_title("AVB", y=1.01, fontsize=18, fontname='sans-serif')
ax1.set_xlabel("Time (s)", fontsize=14, fontname='sans-serif')
ax1.set_ylabel("ΔF/F", fontsize=14, fontname='sans-serif')

ax1.plot(xpos1, AVB_Adult_comp_mean, color='#1F77B4', alpha=0.7, linestyle='dashed', linewidth=2.0, label="Adult")
ax1.plot(xpos2, AVB_Dauer_comp_mean, color='#FF7F0E', alpha=0.7, linestyle='dashed', linewidth=2.0, label="Dauer")

ax1.fill_between(xpos1, AVB_Adult_comp_mean-AVB_Adult_comp_sem/2, AVB_Adult_comp_mean+AVB_Adult_comp_sem/2,\
                 alpha=0.3, edgecolor=None, facecolor='#1F77B4', linewidth=0, linestyle='dashdot')
ax1.fill_between(xpos2, AVB_Dauer_comp_mean-AVB_Dauer_comp_sem/2, AVB_Dauer_comp_mean+AVB_Dauer_comp_sem/2,\
                 alpha=0.3, edgecolor=None, facecolor='#FF7F0E', linewidth=0, linestyle='dashdot')

ax1.axvspan(50, 100, alpha=0.15, color='tab:red')

ax1.legend(bbox_to_anchor=(1, 1.01), loc='upper left', fontsize=14)
ax1.set_xlim(-5,205)
ax1.set_ylim(-0.035,0.012)
ax1.set_xticks(np.arange(0,225,25))
ax1.set_xticklabels(["0","2.5","5.0","7.5","10.0","12.5","15.0","17.5","20.0"], fontsize=11.5, fontname='sans-serif', rotation=0)

# plt.savefig(r'D:\AVB_signal_comp_both_260202.svg', dpi=300, facecolor='w', bbox_inches='tight')
plt.show()


In [ ]:
# plot - AVD :: manuscript ver.

    # svg ; change color ; change font


from matplotlib.ticker import FormatStrFormatter


fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,4))

xpos1 = np.arange(1,201,1)
xpos2 = np.arange(1,201,1)
ax1.set_title("AVD", y=1.01, fontsize=18, fontname='sans-serif')
ax1.set_xlabel("Time (s)", fontsize=14, fontname='sans-serif')
ax1.set_ylabel("ΔF/F", fontsize=14, fontname='sans-serif')

ax1.plot(xpos1, AVD_Adult_comp_mean, color='#1F77B4', alpha=0.7, linestyle='dashed', linewidth=2.0, label="Adult")
ax1.plot(xpos2, AVD_Dauer_comp_mean, color='#FF7F0E', alpha=0.7, linestyle='dashed', linewidth=2.0, label="Dauer")

ax1.fill_between(xpos1, AVD_Adult_comp_mean-AVD_Adult_comp_sem/2, AVD_Adult_comp_mean+AVD_Adult_comp_sem/2,\
                 alpha=0.3, edgecolor=None, facecolor='#1F77B4', linewidth=0, linestyle='dashdot')
ax1.fill_between(xpos2, AVD_Dauer_comp_mean-AVD_Dauer_comp_sem/2, AVD_Dauer_comp_mean+AVD_Dauer_comp_sem/2,\
                 alpha=0.3, edgecolor=None, facecolor='#FF7F0E', linewidth=0, linestyle='dashdot')
ax1.axvspan(50, 100, alpha=0.15, color='tab:red')

ax1.legend(bbox_to_anchor=(1, 1.01), loc='upper left', fontsize=14)
ax1.set_xlim(-5,205)
ax1.set_ylim(-0.009,0.053)
ax1.set_xticks(np.arange(0,225,25))
ax1.set_xticklabels(["0","2.5","5.0","7.5","10.0","12.5","15.0","17.5","20.0"], fontsize=11.5, fontname='sans-serif', rotation=0)

# plt.savefig(r'D:\AVD_signal_comp_both_260202.svg', dpi=300, facecolor='w', bbox_inches='tight')
plt.show()


### signal with Cx36

In [ ]:
# plot - ASH with ASH=AVD Cx36 :: manuscript ver.

    # svg ; change color ; change font



fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,4))

xpos = np.arange(1,201,1)
ax1.set_title("ASH", y=1.01, fontsize=18, fontname='sans-serif')
ax1.set_xlabel("Time (s)", fontsize=14, fontname='sans-serif')
ax1.set_ylabel("ΔF/F", fontsize=14, fontname='sans-serif')

ax1.plot(xpos, ASH_AVD_ASH_Adult_ctrl_mean, color='#1F77B4', alpha=0.6, linestyle='dashed', linewidth=2.0, label="Adult")
ax1.plot(xpos, ASH_AVD_ASH_Dauer_ctrl_mean, color='#FF7F0E', alpha=0.6, linestyle='dashed', linewidth=2.0, label="Dauer")
ax1.plot(xpos, ASH_AVD_ASH_Adult_comp_mean, color='#8C564B', alpha=0.6, linestyle='dashed', linewidth=2.0, label="Adult + ASH=AVD Cx36")

ax1.fill_between(xpos, ASH_AVD_ASH_Adult_ctrl_mean-ASH_AVD_ASH_Adult_ctrl_sem/2, ASH_AVD_ASH_Adult_ctrl_mean+ASH_AVD_ASH_Adult_ctrl_sem/2,\
                 alpha=0.3, edgecolor=None, facecolor='#1F77B4', linewidth=0, linestyle='dashdot')
ax1.fill_between(xpos, ASH_AVD_ASH_Dauer_ctrl_mean-ASH_AVD_ASH_Dauer_ctrl_sem/2, ASH_AVD_ASH_Dauer_ctrl_mean+ASH_AVD_ASH_Dauer_ctrl_sem/2,\
                 alpha=0.3, edgecolor=None, facecolor='#FF7F0E', linewidth=0, linestyle='dashdot')
ax1.fill_between(xpos, ASH_AVD_ASH_Adult_comp_mean-ASH_AVD_ASH_Adult_comp_sem/2, ASH_AVD_ASH_Adult_comp_mean+ASH_AVD_ASH_Adult_comp_sem/2,\
                 alpha=0.3, edgecolor=None, facecolor='#8C564B', linewidth=0, linestyle='dashdot')

ax1.axvspan(50, 100, alpha=0.15, color='tab:red')

ax1.legend(bbox_to_anchor=(1.02, 1.0), loc='upper left', fontsize=14, ncol=1)
ax1.set_xlim(-5,205)
ax1.set_ylim(-0.012,0.042)
ax1.set_xticks(np.arange(0,225,50))
ax1.set_xticklabels(["0","5","10","15","20"], fontsize=11.5, fontname='sans-serif', rotation=0)
# ax1.set_yticks(np.arange(0.2,1.3,0.2))
# ax1.set_yticklabels(["0.2","0.4","0.6","0.8","1.0","1.2"], fontsize=12.5, fontname='sans-serif', rotation=0)
# ax1.yaxis.set_major_formatter(FormatStrFormatter('%.3f'))

# plt.savefig(r'D:\ASH-AVD_ASH_signal_comp_both_260131.svg', dpi=300, facecolor='w', bbox_inches='tight')
plt.show()


In [ ]:
from scipy.stats import mannwhitneyu
from scipy.stats import ranksums

## Curve fitting

In [ ]:
from scipy.optimize import curve_fit

signal_x = np.arange(0,20,0.1)

In [ ]:
# ASH

    # ASH_Adult_comp_mean_adj
    # ASH_Dauer_comp_mean_adj

ASH_Adult_y = np.copy(ASH_Adult_comp_mean_adj)
ASH_Dauer_y = np.copy(ASH_Dauer_comp_mean_adj)


In [ ]:
# fitting function - One-phase decay fitting

    # F(t) = A * e^(-t/tau) + C


def one_phase_decay(X, A, b, C):
    return A * np.exp(-X/b) + C


### ASH=AVD ; ASH

#### Adult with Cx36

In [ ]:
# fitting - ASH, Adult with Cx36

    # check X range

ASH_AVD_ASH_cx36_onep = []

HDH_cx36_popt, HDH_cx36_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_comp_np[0,100:], method="lm"); ASH_AVD_ASH_cx36_onep.append(HDH_cx36_popt[1])  # append 'b' (time constant)
HDH_cx36_popt, HDH_cx36_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_comp_np[1,100:], method="lm"); ASH_AVD_ASH_cx36_onep.append(HDH_cx36_popt[1])
HDH_cx36_popt, HDH_cx36_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_comp_np[2,100:], method="lm"); ASH_AVD_ASH_cx36_onep.append(HDH_cx36_popt[1])
HDH_cx36_popt, HDH_cx36_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_comp_np[3,100:], method="lm"); ASH_AVD_ASH_cx36_onep.append(HDH_cx36_popt[1])
HDH_cx36_popt, HDH_cx36_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_comp_np[4,100:], method="lm"); ASH_AVD_ASH_cx36_onep.append(HDH_cx36_popt[1])
HDH_cx36_popt, HDH_cx36_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_comp_np[5,100:], method="lm"); ASH_AVD_ASH_cx36_onep.append(HDH_cx36_popt[1])
HDH_cx36_popt, HDH_cx36_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_comp_np[6,100:], method="lm"); ASH_AVD_ASH_cx36_onep.append(HDH_cx36_popt[1])
HDH_cx36_popt, HDH_cx36_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_comp_np[7,100:], method="lm"); ASH_AVD_ASH_cx36_onep.append(HDH_cx36_popt[1])
HDH_cx36_popt, HDH_cx36_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_comp_np[8,100:], method="lm"); ASH_AVD_ASH_cx36_onep.append(HDH_cx36_popt[1])
HDH_cx36_popt, HDH_cx36_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_comp_np[9,100:], method="lm"); ASH_AVD_ASH_cx36_onep.append(HDH_cx36_popt[1])
HDH_cx36_popt, HDH_cx36_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_comp_np[10,100:], method="lm"); ASH_AVD_ASH_cx36_onep.append(HDH_cx36_popt[1])
HDH_cx36_popt, HDH_cx36_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_comp_np[11,100:], method="lm"); ASH_AVD_ASH_cx36_onep.append(HDH_cx36_popt[1])
HDH_cx36_popt, HDH_cx36_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_comp_np[12,100:], method="lm"); ASH_AVD_ASH_cx36_onep.append(HDH_cx36_popt[1])
HDH_cx36_popt, HDH_cx36_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_comp_np[13,100:], method="lm"); ASH_AVD_ASH_cx36_onep.append(HDH_cx36_popt[1])
HDH_cx36_popt, HDH_cx36_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_comp_np[14,100:], method="lm"); ASH_AVD_ASH_cx36_onep.append(HDH_cx36_popt[1])
HDH_cx36_popt, HDH_cx36_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_comp_np[15,100:], method="lm"); ASH_AVD_ASH_cx36_onep.append(HDH_cx36_popt[1])
HDH_cx36_popt, HDH_cx36_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_comp_np[16,100:], method="lm"); ASH_AVD_ASH_cx36_onep.append(HDH_cx36_popt[1])
HDH_cx36_popt, HDH_cx36_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_comp_np[17,100:], method="lm"); ASH_AVD_ASH_cx36_onep.append(HDH_cx36_popt[1])

print("ASH Adult Cx36: ", len(ASH_AVD_ASH_cx36_onep), "\n", ASH_AVD_ASH_cx36_onep)


#### ASH Adult ctrl

In [ ]:
# fitting - ASH, Adult ctrl

    # check X range

ASH_AVD_ASH_Adult_ctrl_onep = []

HDH_Adult_ctrl_popt, HDH_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_ctrl_np[0,100:], method="lm"); ASH_AVD_ASH_Adult_ctrl_onep.append(HDH_Adult_ctrl_popt[1])
HDH_Adult_ctrl_popt, HDH_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_ctrl_np[1,100:], method="lm"); ASH_AVD_ASH_Adult_ctrl_onep.append(HDH_Adult_ctrl_popt[1])
HDH_Adult_ctrl_popt, HDH_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_ctrl_np[2,100:], method="lm"); ASH_AVD_ASH_Adult_ctrl_onep.append(HDH_Adult_ctrl_popt[1])
HDH_Adult_ctrl_popt, HDH_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_ctrl_np[3,100:], method="lm"); ASH_AVD_ASH_Adult_ctrl_onep.append(HDH_Adult_ctrl_popt[1])
HDH_Adult_ctrl_popt, HDH_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_ctrl_np[4,100:], method="lm"); ASH_AVD_ASH_Adult_ctrl_onep.append(HDH_Adult_ctrl_popt[1])
HDH_Adult_ctrl_popt, HDH_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_ctrl_np[5,100:], method="lm"); ASH_AVD_ASH_Adult_ctrl_onep.append(HDH_Adult_ctrl_popt[1])
HDH_Adult_ctrl_popt, HDH_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_ctrl_np[6,100:], method="lm"); ASH_AVD_ASH_Adult_ctrl_onep.append(HDH_Adult_ctrl_popt[1])
HDH_Adult_ctrl_popt, HDH_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_ctrl_np[7,100:], method="lm"); ASH_AVD_ASH_Adult_ctrl_onep.append(HDH_Adult_ctrl_popt[1])
HDH_Adult_ctrl_popt, HDH_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_ctrl_np[8,100:], method="lm"); ASH_AVD_ASH_Adult_ctrl_onep.append(HDH_Adult_ctrl_popt[1])
HDH_Adult_ctrl_popt, HDH_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_ctrl_np[9,100:], method="lm"); ASH_AVD_ASH_Adult_ctrl_onep.append(HDH_Adult_ctrl_popt[1])
HDH_Adult_ctrl_popt, HDH_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Adult_ctrl_np[10,100:], method="lm"); ASH_AVD_ASH_Adult_ctrl_onep.append(HDH_Adult_ctrl_popt[1])

print("ASH Adult ctrl: ", len(ASH_AVD_ASH_Adult_ctrl_onep), "\n", ASH_AVD_ASH_Adult_ctrl_onep)


#### ASH Dauer ctrl

In [ ]:
# fitting - ASH, Dauer ctrl

    # check X range

ASH_AVD_ASH_Dauer_ctrl_onep = []

HDH_Dauer_ctrl_popt, HDH_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Dauer_ctrl_np[0,100:], method="lm"); ASH_AVD_ASH_Dauer_ctrl_onep.append(HDH_Dauer_ctrl_popt[1])
HDH_Dauer_ctrl_popt, HDH_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Dauer_ctrl_np[1,100:], method="lm"); ASH_AVD_ASH_Dauer_ctrl_onep.append(HDH_Dauer_ctrl_popt[1])
HDH_Dauer_ctrl_popt, HDH_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Dauer_ctrl_np[2,100:], method="lm"); ASH_AVD_ASH_Dauer_ctrl_onep.append(HDH_Dauer_ctrl_popt[1])
HDH_Dauer_ctrl_popt, HDH_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Dauer_ctrl_np[3,100:], method="lm"); ASH_AVD_ASH_Dauer_ctrl_onep.append(HDH_Dauer_ctrl_popt[1])
HDH_Dauer_ctrl_popt, HDH_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Dauer_ctrl_np[4,100:], method="lm"); ASH_AVD_ASH_Dauer_ctrl_onep.append(HDH_Dauer_ctrl_popt[1])
HDH_Dauer_ctrl_popt, HDH_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Dauer_ctrl_np[5,100:], method="lm"); ASH_AVD_ASH_Dauer_ctrl_onep.append(HDH_Dauer_ctrl_popt[1])
HDH_Dauer_ctrl_popt, HDH_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Dauer_ctrl_np[6,100:], method="lm"); ASH_AVD_ASH_Dauer_ctrl_onep.append(HDH_Dauer_ctrl_popt[1])
HDH_Dauer_ctrl_popt, HDH_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Dauer_ctrl_np[7,100:], method="lm"); ASH_AVD_ASH_Dauer_ctrl_onep.append(HDH_Dauer_ctrl_popt[1])
HDH_Dauer_ctrl_popt, HDH_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Dauer_ctrl_np[8,100:], method="lm"); ASH_AVD_ASH_Dauer_ctrl_onep.append(HDH_Dauer_ctrl_popt[1])
HDH_Dauer_ctrl_popt, HDH_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Dauer_ctrl_np[9,100:], method="lm"); ASH_AVD_ASH_Dauer_ctrl_onep.append(HDH_Dauer_ctrl_popt[1])
HDH_Dauer_ctrl_popt, HDH_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Dauer_ctrl_np[10,100:], method="lm"); ASH_AVD_ASH_Dauer_ctrl_onep.append(HDH_Dauer_ctrl_popt[1])
HDH_Dauer_ctrl_popt, HDH_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[100:], ASH_AVD_ASH_Dauer_ctrl_np[11,100:], method="lm"); ASH_AVD_ASH_Dauer_ctrl_onep.append(HDH_Dauer_ctrl_popt[1])

print("ASH Dauer ctrl: ", len(ASH_AVD_ASH_Dauer_ctrl_onep), "\n", ASH_AVD_ASH_Dauer_ctrl_onep)


In [ ]:
print("ASH; Adult ctrl vs. Adult cx36 : ", stats.ranksums(ASH_AVD_ASH_Adult_ctrl_onep, ASH_AVD_ASH_cx36_onep)[1])
print("ASH; Adult ctrl vs. Dauer ctrl : ", stats.ranksums(ASH_AVD_ASH_Adult_ctrl_onep, ASH_AVD_ASH_Dauer_ctrl_onep)[1])
print("ASH; Dauer ctrl vs. Adult cx36 : ", stats.ranksums(ASH_AVD_ASH_Dauer_ctrl_onep, ASH_AVD_ASH_cx36_onep)[1])

In [ ]:
# plot


b_width = 0.5; v_width = 0.6; ssz=14; fsz=14


fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(3.5,4))

ax1.set_title("ASH", x=0.5, y=1.01, fontsize=18, fontname='sans-serif')
ax1.set_ylabel("Time (s)", fontsize=14, fontname='sans-serif')
xpos = [1,2,3]

ax1.boxplot(ASH_AVD_ASH_Adult_ctrl_onep, positions=[xpos[0]], widths=b_width, patch_artist=True, whis=(5,95),\
            showmeans=False, showfliers=False, medianprops={"color": "k", "linewidth": 1.0},\
            boxprops={"facecolor": ('#1F77B4'), "edgecolor": "k", "linewidth": 1.6, "alpha": 0.6},\
            whiskerprops={"color": "k", "linewidth": 1.6}, capprops={"color": "k", "linewidth": 1.6})
ax1.boxplot(ASH_AVD_ASH_Dauer_ctrl_onep, positions=[xpos[2]], widths=b_width, patch_artist=True, whis=(5,95),\
            showmeans=False, showfliers=False, medianprops={"color": "k", "linewidth": 1.0},\
            boxprops={"facecolor": ('#FF7F0E'), "edgecolor": "k", "linewidth": 1.6, "alpha": 0.6},\
            whiskerprops={"color": "k", "linewidth": 1.6}, capprops={"color": "k", "linewidth": 1.6})
ax1.boxplot(ASH_AVD_ASH_cx36_onep, positions=[xpos[1]], widths=b_width, patch_artist=True, whis=(5,95),\
            showmeans=False, showfliers=False, medianprops={"color": "k", "linewidth": 1.0},\
            boxprops={"facecolor": ('#8C564B'), "edgecolor": "k", "linewidth": 1.6, "alpha": 0.6},\
            whiskerprops={"color": "k", "linewidth": 1.6}, capprops={"color": "k", "linewidth": 1.6})

ax1.set_xlim(0.3,3.7)
ax1.set_ylim(-0.09,4.50)
label = ["Cx36 X\nAdult","Cx36 O\nAdult","\nDauer"]
ax1.set_xticks(xpos)
ax1.set_xticklabels(label, fontsize=fsz, rotation=0, fontname='sans-serif')

ax1.plot([0.9,3.1], [3.8,3.8], lw=0.8, c='k')
ax1.text(2.0, 3.9, "n.s", ha='center', va='bottom', color='k', fontsize=14, fontname='sans-serif')
fig.tight_layout(pad=0.4)

plt.show()


### ASH=AVD ; AVD

In [ ]:
# fitting function - One-phase rise fitting

    # F(t) = A * e^(t/tau) + C


def one_phase_rise(X, A, b, C):
    return A * np.exp(X/b) + C


#### Adult with Cx36

In [ ]:
# normalize

ASH_AVD_AVD_Adult_comp_np_norm = np.zeros(ASH_AVD_AVD_Adult_comp_np.shape)


for ii in range(ASH_AVD_AVD_Adult_comp_np_norm.shape[0]):
    
    SIGNAL = np.copy(ASH_AVD_AVD_Adult_comp_np[ii,:])
    MIN = np.min(SIGNAL); MAX = np.max(SIGNAL)
    
    ASH_AVD_AVD_Adult_comp_np_norm[ii,:] = (SIGNAL - MIN) / (MAX - MIN)


In [ ]:
# fitting - AVD, Adult with Cx36

    # check X range

ASH_AVD_AVD_cx36_onep_norm = []

HDD_cx36_popt, HDD_cx36_pcov = curve_fit(one_phase_rise, signal_x[25:55], ASH_AVD_AVD_Adult_comp_np_norm[0,25:55], method="lm"); ASH_AVD_AVD_cx36_onep_norm.append(HDD_cx36_popt[1])  # append 'b' (time constant)
HDD_cx36_popt, HDD_cx36_pcov = curve_fit(one_phase_rise, signal_x[10:55], ASH_AVD_AVD_Adult_comp_np_norm[1,10:55], method="lm"); ASH_AVD_AVD_cx36_onep_norm.append(HDD_cx36_popt[1])
HDD_cx36_popt, HDD_cx36_pcov = curve_fit(one_phase_rise, signal_x[1:70], ASH_AVD_AVD_Adult_comp_np_norm[2,1:70], method="lm"); ASH_AVD_AVD_cx36_onep_norm.append(HDD_cx36_popt[1])
HDD_cx36_popt, HDD_cx36_pcov = curve_fit(one_phase_rise, signal_x[15:71], ASH_AVD_AVD_Adult_comp_np_norm[3,15:71], method="lm"); ASH_AVD_AVD_cx36_onep_norm.append(HDD_cx36_popt[1])
HDD_cx36_popt, HDD_cx36_pcov = curve_fit(one_phase_rise, signal_x[35:82], ASH_AVD_AVD_Adult_comp_np_norm[4,35:82], method="lm"); ASH_AVD_AVD_cx36_onep_norm.append(HDD_cx36_popt[1])
HDD_cx36_popt, HDD_cx36_pcov = curve_fit(one_phase_rise, signal_x[30:100], ASH_AVD_AVD_Adult_comp_np_norm[5,30:100], method="lm"); ASH_AVD_AVD_cx36_onep_norm.append(HDD_cx36_popt[1])
HDD_cx36_popt, HDD_cx36_pcov = curve_fit(one_phase_rise, signal_x[45:100], ASH_AVD_AVD_Adult_comp_np_norm[6,45:100], method="lm"); ASH_AVD_AVD_cx36_onep_norm.append(HDD_cx36_popt[1])
HDD_cx36_popt, HDD_cx36_pcov = curve_fit(one_phase_rise, signal_x[27:70], ASH_AVD_AVD_Adult_comp_np_norm[7,27:70], method="lm"); ASH_AVD_AVD_cx36_onep_norm.append(HDD_cx36_popt[1])
HDD_cx36_popt, HDD_cx36_pcov = curve_fit(one_phase_rise, signal_x[15:75], ASH_AVD_AVD_Adult_comp_np_norm[8,15:75], method="lm"); ASH_AVD_AVD_cx36_onep_norm.append(HDD_cx36_popt[1])
HDD_cx36_popt, HDD_cx36_pcov = curve_fit(one_phase_rise, signal_x[12:55], ASH_AVD_AVD_Adult_comp_np_norm[9,12:55], method="lm"); ASH_AVD_AVD_cx36_onep_norm.append(HDD_cx36_popt[1])
HDD_cx36_popt, HDD_cx36_pcov = curve_fit(one_phase_rise, signal_x[48:80], ASH_AVD_AVD_Adult_comp_np_norm[10,48:80], method="lm"); ASH_AVD_AVD_cx36_onep_norm.append(HDD_cx36_popt[1])
HDD_cx36_popt, HDD_cx36_pcov = curve_fit(one_phase_rise, signal_x[50:80], ASH_AVD_AVD_Adult_comp_np_norm[11,50:80], method="lm"); ASH_AVD_AVD_cx36_onep_norm.append(HDD_cx36_popt[1])
HDD_cx36_popt, HDD_cx36_pcov = curve_fit(one_phase_rise, signal_x[35:65], ASH_AVD_AVD_Adult_comp_np_norm[12,35:65], method="lm"); ASH_AVD_AVD_cx36_onep_norm.append(HDD_cx36_popt[1])

print("AVD Adult Cx36: ", len(ASH_AVD_AVD_cx36_onep_norm), "\n", ASH_AVD_AVD_cx36_onep_norm)


#### AVD Adult ctrl

In [ ]:
# normalize

ASH_AVD_AVD_Adult_ctrl_np_norm = np.zeros(ASH_AVD_AVD_Adult_ctrl_np.shape)


for ii in range(ASH_AVD_AVD_Adult_ctrl_np_norm.shape[0]):
    
    SIGNAL = np.copy(ASH_AVD_AVD_Adult_ctrl_np[ii,:])
    MIN = np.min(SIGNAL); MAX = np.max(SIGNAL)
    
    ASH_AVD_AVD_Adult_ctrl_np_norm[ii,:] = (SIGNAL - MIN) / (MAX - MIN)


In [ ]:
# fitting - AVD, Adult ctrl

    # check X range

ASH_AVD_AVD_Adult_ctrl_onep_norm = []

HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[15:105], ASH_AVD_AVD_Adult_ctrl_np_norm[0,15:105], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])  # append 'b' (time constant)
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[20:75], ASH_AVD_AVD_Adult_ctrl_np_norm[1,20:75], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[25:90], ASH_AVD_AVD_Adult_ctrl_np_norm[2,25:90], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[30:110], ASH_AVD_AVD_Adult_ctrl_np_norm[3,30:110], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[50:145], ASH_AVD_AVD_Adult_ctrl_np_norm[4,50:145], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[120:185], ASH_AVD_AVD_Adult_ctrl_np_norm[5,120:185], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[60:150], ASH_AVD_AVD_Adult_ctrl_np_norm[6,60:150], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[50:110], ASH_AVD_AVD_Adult_ctrl_np_norm[7,50:110], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[45:100], ASH_AVD_AVD_Adult_ctrl_np_norm[8,45:100], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[25:125], ASH_AVD_AVD_Adult_ctrl_np_norm[9,25:125], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[25:100], ASH_AVD_AVD_Adult_ctrl_np_norm[10,25:100], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[40:85], ASH_AVD_AVD_Adult_ctrl_np_norm[11,40:85], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[50:120], ASH_AVD_AVD_Adult_ctrl_np_norm[12,50:120], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[35:90], ASH_AVD_AVD_Adult_ctrl_np_norm[13,35:90], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[25:100], ASH_AVD_AVD_Adult_ctrl_np_norm[14,25:100], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[25:130], ASH_AVD_AVD_Adult_ctrl_np_norm[15,25:130], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[40:90], ASH_AVD_AVD_Adult_ctrl_np_norm[16,40:90], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[50:125], ASH_AVD_AVD_Adult_ctrl_np_norm[17,50:125], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[25:100], ASH_AVD_AVD_Adult_ctrl_np_norm[18,25:100], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[10:70], ASH_AVD_AVD_Adult_ctrl_np_norm[19,10:70], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[10:80], ASH_AVD_AVD_Adult_ctrl_np_norm[20,10:80], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[15:125], ASH_AVD_AVD_Adult_ctrl_np_norm[21,15:125], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[5:95], ASH_AVD_AVD_Adult_ctrl_np_norm[22,5:95], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[45:175], ASH_AVD_AVD_Adult_ctrl_np_norm[23,45:175], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[20:72], ASH_AVD_AVD_Adult_ctrl_np_norm[24,20:72], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[42:70], ASH_AVD_AVD_Adult_ctrl_np_norm[25,42:70], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[25:125], ASH_AVD_AVD_Adult_ctrl_np_norm[26,25:125], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[30:110], ASH_AVD_AVD_Adult_ctrl_np_norm[27,30:110], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[50:130], ASH_AVD_AVD_Adult_ctrl_np_norm[28,50:130], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[0:72], ASH_AVD_AVD_Adult_ctrl_np_norm[29,0:72], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[40:135], ASH_AVD_AVD_Adult_ctrl_np_norm[30,40:135], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[40:165], ASH_AVD_AVD_Adult_ctrl_np_norm[31,40:165], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[20:135], ASH_AVD_AVD_Adult_ctrl_np_norm[32,20:135], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[50:170], ASH_AVD_AVD_Adult_ctrl_np_norm[33,50:170], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[5:115], ASH_AVD_AVD_Adult_ctrl_np_norm[34,5:115], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[25:90], ASH_AVD_AVD_Adult_ctrl_np_norm[35,25:90], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[50:105], ASH_AVD_AVD_Adult_ctrl_np_norm[36,50:105], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[45:110], ASH_AVD_AVD_Adult_ctrl_np_norm[37,45:110], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[10:130], ASH_AVD_AVD_Adult_ctrl_np_norm[38,10:130], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[10:110], ASH_AVD_AVD_Adult_ctrl_np_norm[39,10:110], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[25:175], ASH_AVD_AVD_Adult_ctrl_np_norm[40,25:175], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[45:90], ASH_AVD_AVD_Adult_ctrl_np_norm[41,45:90], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[15:125], ASH_AVD_AVD_Adult_ctrl_np_norm[42,15:125], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])
HDD_Adult_ctrl_popt, HDD_Adult_ctrl_pcov = curve_fit(one_phase_rise, signal_x[0:70], ASH_AVD_AVD_Adult_ctrl_np_norm[43,0:70], method="lm"); ASH_AVD_AVD_Adult_ctrl_onep_norm.append(HDD_Adult_ctrl_popt[1])

print("AVD Adult ctrl: ", len(ASH_AVD_AVD_Adult_ctrl_onep_norm), "\n", ASH_AVD_AVD_Adult_ctrl_onep_norm)


#### AVD Dauer ctrl

In [ ]:
# normalize

ASH_AVD_AVD_Dauer_ctrl_np_norm = np.zeros(ASH_AVD_AVD_Dauer_ctrl_np.shape)


for ii in range(ASH_AVD_AVD_Dauer_ctrl_np_norm.shape[0]):
    
    SIGNAL = np.copy(ASH_AVD_AVD_Dauer_ctrl_np[ii,:])
    MIN = np.min(SIGNAL); MAX = np.max(SIGNAL)
    
    ASH_AVD_AVD_Dauer_ctrl_np_norm[ii,:] = (SIGNAL - MIN) / (MAX - MIN)


In [ ]:
# fitting - AVD, Dauer ctrl

    # check X range

ASH_AVD_AVD_Dauer_ctrl_onep_norm = []

HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[10:50], ASH_AVD_AVD_Dauer_ctrl_np_norm[0,10:50], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])  # append 'b' (time constant)
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[100:130], ASH_AVD_AVD_Dauer_ctrl_np_norm[1,100:130], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[20:120], ASH_AVD_AVD_Dauer_ctrl_np_norm[2,20:120], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[25:110], ASH_AVD_AVD_Dauer_ctrl_np_norm[3,25:110], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[0:60], ASH_AVD_AVD_Dauer_ctrl_np_norm[4,0:60], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[50:62], ASH_AVD_AVD_Dauer_ctrl_np_norm[5,50:62], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[35:130], ASH_AVD_AVD_Dauer_ctrl_np_norm[6,35:130], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[15:60], ASH_AVD_AVD_Dauer_ctrl_np_norm[7,15:60], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[30:80], ASH_AVD_AVD_Dauer_ctrl_np_norm[8,30:80], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[40:75], ASH_AVD_AVD_Dauer_ctrl_np_norm[9,40:75], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[40:97], ASH_AVD_AVD_Dauer_ctrl_np_norm[10,40:97], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[55:105], ASH_AVD_AVD_Dauer_ctrl_np_norm[11,55:105], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[60:80], ASH_AVD_AVD_Dauer_ctrl_np_norm[12,60:80], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[30:67], ASH_AVD_AVD_Dauer_ctrl_np_norm[13,30:67], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[30:72], ASH_AVD_AVD_Dauer_ctrl_np_norm[14,30:72], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[37:60], ASH_AVD_AVD_Dauer_ctrl_np_norm[15,37:60], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[51:83], ASH_AVD_AVD_Dauer_ctrl_np_norm[16,51:83], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[77:120], ASH_AVD_AVD_Dauer_ctrl_np_norm[17,77:120], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[23:60], ASH_AVD_AVD_Dauer_ctrl_np_norm[18,23:60], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[55:120], ASH_AVD_AVD_Dauer_ctrl_np_norm[19,55:120], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[20:55], ASH_AVD_AVD_Dauer_ctrl_np_norm[20,20:55], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[37:67], ASH_AVD_AVD_Dauer_ctrl_np_norm[21,37:67], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[75:110], ASH_AVD_AVD_Dauer_ctrl_np_norm[22,75:110], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[35:72], ASH_AVD_AVD_Dauer_ctrl_np_norm[23,35:72], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[73:118], ASH_AVD_AVD_Dauer_ctrl_np_norm[24,73:118], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[25:70], ASH_AVD_AVD_Dauer_ctrl_np_norm[25,25:70], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[52:100], ASH_AVD_AVD_Dauer_ctrl_np_norm[26,52:100], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[73:95], ASH_AVD_AVD_Dauer_ctrl_np_norm[27,73:95], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[20:80], ASH_AVD_AVD_Dauer_ctrl_np_norm[28,20:80], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[25:75], ASH_AVD_AVD_Dauer_ctrl_np_norm[29,25:75], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[40:90], ASH_AVD_AVD_Dauer_ctrl_np_norm[30,40:90], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[0:100], ASH_AVD_AVD_Dauer_ctrl_np_norm[31,0:100], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[45:80], ASH_AVD_AVD_Dauer_ctrl_np_norm[32,45:80], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[30:55], ASH_AVD_AVD_Dauer_ctrl_np_norm[33,30:55], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[10:65], ASH_AVD_AVD_Dauer_ctrl_np_norm[34,10:65], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[0:70], ASH_AVD_AVD_Dauer_ctrl_np_norm[35,0:70], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[0:70], ASH_AVD_AVD_Dauer_ctrl_np_norm[36,0:70], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[27:72], ASH_AVD_AVD_Dauer_ctrl_np_norm[37,27:72], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[30:67], ASH_AVD_AVD_Dauer_ctrl_np_norm[38,30:67], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[33:65], ASH_AVD_AVD_Dauer_ctrl_np_norm[39,33:65], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[50:95], ASH_AVD_AVD_Dauer_ctrl_np_norm[40,50:95], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[105:127], ASH_AVD_AVD_Dauer_ctrl_np_norm[41,105:127], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[37:70], ASH_AVD_AVD_Dauer_ctrl_np_norm[42,37:70], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[40:95], ASH_AVD_AVD_Dauer_ctrl_np_norm[43,40:95], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[32:95], ASH_AVD_AVD_Dauer_ctrl_np_norm[44,32:95], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[20:53], ASH_AVD_AVD_Dauer_ctrl_np_norm[45,20:53], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[27:73], ASH_AVD_AVD_Dauer_ctrl_np_norm[46,27:73], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[50:115], ASH_AVD_AVD_Dauer_ctrl_np_norm[47,50:115], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[0:55], ASH_AVD_AVD_Dauer_ctrl_np_norm[48,0:55], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[5:55], ASH_AVD_AVD_Dauer_ctrl_np_norm[49,5:55], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[25:65], ASH_AVD_AVD_Dauer_ctrl_np_norm[50,25:65], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[19:52], ASH_AVD_AVD_Dauer_ctrl_np_norm[51,19:52], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[12:60], ASH_AVD_AVD_Dauer_ctrl_np_norm[52,12:60], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[35:90], ASH_AVD_AVD_Dauer_ctrl_np_norm[53,35:90], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[10:77], ASH_AVD_AVD_Dauer_ctrl_np_norm[54,10:77], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[1:80], ASH_AVD_AVD_Dauer_ctrl_np_norm[55,1:80], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[20:80], ASH_AVD_AVD_Dauer_ctrl_np_norm[56,20:80], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[25:55], ASH_AVD_AVD_Dauer_ctrl_np_norm[57,25:55], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[25:70], ASH_AVD_AVD_Dauer_ctrl_np_norm[58,25:70], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[27:70], ASH_AVD_AVD_Dauer_ctrl_np_norm[59,27:70], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])
HDD_Dauer_ctrl_popt, HDD_Dauer_ctrl_pcov = curve_fit(one_phase_rise, signal_x[45:85], ASH_AVD_AVD_Dauer_ctrl_np_norm[60,45:85], method="lm"); ASH_AVD_AVD_Dauer_ctrl_onep_norm.append(HDD_Dauer_ctrl_popt[1])

print("AVD Dauer ctrl: ", len(ASH_AVD_AVD_Dauer_ctrl_onep_norm), "\n", ASH_AVD_AVD_Dauer_ctrl_onep_norm)


---

In [ ]:
print("AVD; Adult ctrl vs. Adult cx36 : ", stats.ranksums(ASH_AVD_AVD_Adult_ctrl_onep_norm, ASH_AVD_AVD_cx36_onep_norm)[1])
print("AVD; Adult ctrl vs. Dauer ctrl : ", stats.ranksums(ASH_AVD_AVD_Adult_ctrl_onep_norm, ASH_AVD_AVD_Dauer_ctrl_onep_norm)[1])
print("AVD; Dauer ctrl vs. Adult cx36 : ", stats.ranksums(ASH_AVD_AVD_Dauer_ctrl_onep_norm, ASH_AVD_AVD_cx36_onep_norm)[1])

In [ ]:
# plot

b_width = 0.5; v_width = 0.6; ssz=14; fsz=14


fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(3.5,4))

ax1.set_title("AVD", x=0.5, y=1.01, fontsize=18, fontname='sans-serif')
ax1.set_ylabel("Time (s)", fontsize=14, fontname='sans-serif')
xpos = [1,2,3]

ax1.boxplot(ASH_AVD_AVD_Adult_ctrl_onep_norm, positions=[xpos[0]], widths=b_width, patch_artist=True, whis=(5,95),\
            showmeans=False, showfliers=False, medianprops={"color": "k", "linewidth": 1.0},\
            boxprops={"facecolor": ('#1F77B4'), "edgecolor": "k", "linewidth": 1.6, "alpha": 0.6},\
            whiskerprops={"color": "k", "linewidth": 1.6}, capprops={"color": "k", "linewidth": 1.6})
ax1.boxplot(ASH_AVD_AVD_Dauer_ctrl_onep_norm, positions=[xpos[2]], widths=b_width, patch_artist=True, whis=(5,95),\
            showmeans=False, showfliers=False, medianprops={"color": "k", "linewidth": 1.0},\
            boxprops={"facecolor": ('#FF7F0E'), "edgecolor": "k", "linewidth": 1.6, "alpha": 0.6},\
            whiskerprops={"color": "k", "linewidth": 1.6}, capprops={"color": "k", "linewidth": 1.6})
ax1.boxplot(ASH_AVD_AVD_cx36_onep_norm, positions=[xpos[1]], widths=b_width, patch_artist=True, whis=(5,95),\
            showmeans=False, showfliers=False, medianprops={"color": "k", "linewidth": 1.0},\
            boxprops={"facecolor": ('#8C564B'), "edgecolor": "k", "linewidth": 1.6, "alpha": 0.6},\
            whiskerprops={"color": "k", "linewidth": 1.6}, capprops={"color": "k", "linewidth": 1.6})

ax1.set_xlim(0.3,3.7)
label = ["Cx36 X\nAdult","Cx36 O\nAdult","\nDauer"]
ax1.set_xticks(xpos)
ax1.set_xticklabels(label, fontsize=fsz, rotation=0, fontname='sans-serif')

ax1.plot([0.9,2.1], [7.2,7.2], lw=0.8, c='k')
ax1.text(1.5, 7.3, "**", ha='center', va='bottom', color='k', fontsize=14, fontname='sans-serif')
ax1.plot([1.9,3.1], [4.5,4.5], lw=0.8, c='k')
ax1.text(2.5, 4.6, "n.s", ha='center', va='bottom', color='k', fontsize=14, fontname='sans-serif')
fig.tight_layout(pad=0.4)

plt.show()


In [ ]:
# get mean, std, sem (normalized ver.)

from scipy.stats import sem

ASH_AVD_AVD_Adult_comp_norm_mean = np.nanmean(ASH_AVD_AVD_Adult_comp_np_norm, axis=0)
ASH_AVD_AVD_Adult_comp_norm_std = np.nanstd(ASH_AVD_AVD_Adult_comp_np_norm, axis=0)
ASH_AVD_AVD_Adult_comp_norm_sem = sem(ASH_AVD_AVD_Adult_comp_np_norm, nan_policy='omit', axis=0)

ASH_AVD_AVD_Adult_ctrl_norm_mean = np.nanmean(ASH_AVD_AVD_Adult_ctrl_np_norm, axis=0)
ASH_AVD_AVD_Adult_ctrl_norm_std = np.nanstd(ASH_AVD_AVD_Adult_ctrl_np_norm, axis=0)
ASH_AVD_AVD_Adult_ctrl_norm_sem = sem(ASH_AVD_AVD_Adult_ctrl_np_norm, nan_policy='omit', axis=0)

ASH_AVD_AVD_Dauer_ctrl_norm_mean = np.nanmean(ASH_AVD_AVD_Dauer_ctrl_np_norm, axis=0)
ASH_AVD_AVD_Dauer_ctrl_norm_std = np.nanstd(ASH_AVD_AVD_Dauer_ctrl_np_norm, axis=0)
ASH_AVD_AVD_Dauer_ctrl_norm_sem = sem(ASH_AVD_AVD_Dauer_ctrl_np_norm, nan_policy='omit', axis=0)


In [ ]:
# plot

fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,4))

xpos = np.arange(1,201,1)
ax1.set_title("AVD", y=1.01, fontsize=18, fontname='sans-serif')
ax1.set_xlabel("Time (s)", fontsize=14, fontname='sans-serif')
ax1.set_ylabel("Fnorm", fontsize=14, fontname='sans-serif')

ax1.plot(xpos, ASH_AVD_AVD_Adult_ctrl_norm_mean, color='#1F77B4', alpha=0.6, linestyle='dashed', linewidth=2.0, label="Adult")
ax1.plot(xpos, ASH_AVD_AVD_Dauer_ctrl_norm_mean, color='#FF7F0E', alpha=0.6, linestyle='dashed', linewidth=2.0, label="Dauer")
ax1.plot(xpos, ASH_AVD_AVD_Adult_comp_norm_mean, color='#8C564B', alpha=0.6, linestyle='dashed', linewidth=2.0, label="Adult + ASH=AVD Cx36")

ax1.fill_between(xpos, ASH_AVD_AVD_Adult_ctrl_norm_mean-ASH_AVD_AVD_Adult_ctrl_norm_sem/2, ASH_AVD_AVD_Adult_ctrl_norm_mean+ASH_AVD_AVD_Adult_ctrl_norm_sem/2,\
                 alpha=0.3, edgecolor=None, facecolor='#1F77B4', linewidth=0, linestyle='dashdot')
ax1.fill_between(xpos, ASH_AVD_AVD_Dauer_ctrl_norm_mean-ASH_AVD_AVD_Dauer_ctrl_norm_sem/2, ASH_AVD_AVD_Dauer_ctrl_norm_mean+ASH_AVD_AVD_Dauer_ctrl_norm_sem/2,\
                 alpha=0.3, edgecolor=None, facecolor='#FF7F0E', linewidth=0, linestyle='dashdot')
ax1.fill_between(xpos, ASH_AVD_AVD_Adult_comp_norm_mean-ASH_AVD_AVD_Adult_comp_norm_sem/2, ASH_AVD_AVD_Adult_comp_norm_mean+ASH_AVD_AVD_Adult_comp_norm_sem/2,\
                 alpha=0.3, edgecolor=None, facecolor='#8C564B', linewidth=0, linestyle='dashdot')

ax1.axvspan(50, 100, alpha=0.15, color='tab:red')

ax1.legend(bbox_to_anchor=(1.02, 1.0), loc='upper left', fontsize=14, ncol=1)
ax1.set_xlim(-5,205)
ax1.set_ylim(0.12,0.83)
ax1.set_xticks(np.arange(0,225,50))
ax1.set_xticklabels(["0","5","10","15","20"], fontsize=11.5, fontname='sans-serif', rotation=0)

plt.show()


### AVB=AVD ; AVB

#### Adult with Cx36

In [ ]:
# fitting - AVB, Adult with Cx36

    # check X range

AVB_AVD_AVB_cx36_onep = []

BDB_cx36_popt, BDB_cx36_pcov = curve_fit(one_phase_decay, signal_x[62:105], AVB_AVD_AVB_Adult_comp_np[0,62:105], method="lm"); AVB_AVD_AVB_cx36_onep.append(BDB_cx36_popt[1])  # append 'b' (time constant)
BDB_cx36_popt, BDB_cx36_pcov = curve_fit(one_phase_decay, signal_x[48:75], AVB_AVD_AVB_Adult_comp_np[1,48:75], method="lm"); AVB_AVD_AVB_cx36_onep.append(BDB_cx36_popt[1])
BDB_cx36_popt, BDB_cx36_pcov = curve_fit(one_phase_decay, signal_x[62:92], AVB_AVD_AVB_Adult_comp_np[2,62:92], method="lm"); AVB_AVD_AVB_cx36_onep.append(BDB_cx36_popt[1])
BDB_cx36_popt, BDB_cx36_pcov = curve_fit(one_phase_decay, signal_x[79:125], AVB_AVD_AVB_Adult_comp_np[3,79:125], method="lm"); AVB_AVD_AVB_cx36_onep.append(BDB_cx36_popt[1])
BDB_cx36_popt, BDB_cx36_pcov = curve_fit(one_phase_decay, signal_x[75:125], AVB_AVD_AVB_Adult_comp_np[4,75:125], method="lm"); AVB_AVD_AVB_cx36_onep.append(BDB_cx36_popt[1])
BDB_cx36_popt, BDB_cx36_pcov = curve_fit(one_phase_decay, signal_x[50:78], AVB_AVD_AVB_Adult_comp_np[5,50:78], method="lm"); AVB_AVD_AVB_cx36_onep.append(BDB_cx36_popt[1])
BDB_cx36_popt, BDB_cx36_pcov = curve_fit(one_phase_decay, signal_x[49:107], AVB_AVD_AVB_Adult_comp_np[6,49:107], method="lm"); AVB_AVD_AVB_cx36_onep.append(BDB_cx36_popt[1])
BDB_cx36_popt, BDB_cx36_pcov = curve_fit(one_phase_decay, signal_x[52:110], AVB_AVD_AVB_Adult_comp_np[7,52:110], method="lm"); AVB_AVD_AVB_cx36_onep.append(BDB_cx36_popt[1])
BDB_cx36_popt, BDB_cx36_pcov = curve_fit(one_phase_decay, signal_x[47:100], AVB_AVD_AVB_Adult_comp_np[8,47:100], method="lm"); AVB_AVD_AVB_cx36_onep.append(BDB_cx36_popt[1])
BDB_cx36_popt, BDB_cx36_pcov = curve_fit(one_phase_decay, signal_x[146:], AVB_AVD_AVB_Adult_comp_np[9,146:], method="lm"); AVB_AVD_AVB_cx36_onep.append(BDB_cx36_popt[1])
BDB_cx36_popt, BDB_cx36_pcov = curve_fit(one_phase_decay, signal_x[48:75], AVB_AVD_AVB_Adult_comp_np[10,48:75], method="lm"); AVB_AVD_AVB_cx36_onep.append(BDB_cx36_popt[1])
BDB_cx36_popt, BDB_cx36_pcov = curve_fit(one_phase_decay, signal_x[55:78], AVB_AVD_AVB_Adult_comp_np[11,55:78], method="lm"); AVB_AVD_AVB_cx36_onep.append(BDB_cx36_popt[1])

print("AVD Adult Cx36: ", len(AVB_AVD_AVB_cx36_onep), "\n", AVB_AVD_AVB_cx36_onep)


#### AVB Adult ctrl

In [ ]:
# fitting - AVB, Adult ctrl

    # check X range

AVB_AVD_AVB_Adult_ctrl_onep = []

BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[52:95], AVB_AVD_AVB_Adult_ctrl_np[0,52:95], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])  # append 'b' (time constant)
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[70:125], AVB_AVD_AVB_Adult_ctrl_np[1,70:125], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[45:75], AVB_AVD_AVB_Adult_ctrl_np[2,45:75], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[75:130], AVB_AVD_AVB_Adult_ctrl_np[3,75:130], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[105:180], AVB_AVD_AVB_Adult_ctrl_np[4,105:180], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[70:120], AVB_AVD_AVB_Adult_ctrl_np[5,70:120], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[50:105], AVB_AVD_AVB_Adult_ctrl_np[6,50:105], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[60:150], AVB_AVD_AVB_Adult_ctrl_np[7,60:150], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[50:125], AVB_AVD_AVB_Adult_ctrl_np[8,50:125], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[152:181], AVB_AVD_AVB_Adult_ctrl_np[9,152:181], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[75:130], AVB_AVD_AVB_Adult_ctrl_np[10,75:130], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[70:125], AVB_AVD_AVB_Adult_ctrl_np[11,70:125], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[55:125], AVB_AVD_AVB_Adult_ctrl_np[12,55:125], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[95:125], AVB_AVD_AVB_Adult_ctrl_np[13,95:125], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[73:95], AVB_AVD_AVB_Adult_ctrl_np[14,73:95], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[150:180], AVB_AVD_AVB_Adult_ctrl_np[15,150:180], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[105:168], AVB_AVD_AVB_Adult_ctrl_np[16,105:168], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[60:150], AVB_AVD_AVB_Adult_ctrl_np[17,60:150], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[37:120], AVB_AVD_AVB_Adult_ctrl_np[18,37:120], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[105:], AVB_AVD_AVB_Adult_ctrl_np[19,105:], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[95:160], AVB_AVD_AVB_Adult_ctrl_np[20,95:160], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[80:150], AVB_AVD_AVB_Adult_ctrl_np[21,80:150], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[100:180], AVB_AVD_AVB_Adult_ctrl_np[22,100:180], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[100:125], AVB_AVD_AVB_Adult_ctrl_np[23,100:125], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[50:80], AVB_AVD_AVB_Adult_ctrl_np[24,50:80], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[70:95], AVB_AVD_AVB_Adult_ctrl_np[25,70:95], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[95:], AVB_AVD_AVB_Adult_ctrl_np[26,95:], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[40:120], AVB_AVD_AVB_Adult_ctrl_np[27,40:120], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[150:175], AVB_AVD_AVB_Adult_ctrl_np[28,150:175], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[105:150], AVB_AVD_AVB_Adult_ctrl_np[29,105:150], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[85:125], AVB_AVD_AVB_Adult_ctrl_np[30,85:125], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[90:160], AVB_AVD_AVB_Adult_ctrl_np[31,90:160], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[125:], AVB_AVD_AVB_Adult_ctrl_np[32,125:], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])
BDB_Adult_ctrl_popt, BDB_Adult_ctrl_pcov = curve_fit(one_phase_decay, signal_x[48:75], AVB_AVD_AVB_Adult_ctrl_np[33,48:75], method="lm"); AVB_AVD_AVB_Adult_ctrl_onep.append(BDB_Adult_ctrl_popt[1])

print("AVB Adult ctrl: ", len(AVB_AVD_AVB_Adult_ctrl_onep), "\n", AVB_AVD_AVB_Adult_ctrl_onep)


#### AVB Dauer ctrl

In [ ]:
# fitting - AVB, Dauer ctrl

    # check X range

AVB_AVD_AVB_Dauer_ctrl_onep = []

BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[50:69], AVB_AVD_AVB_Dauer_ctrl_np[0,50:69], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])  # append 'b' (time constant)
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[57:82], AVB_AVD_AVB_Dauer_ctrl_np[1,57:82], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[55:77], AVB_AVD_AVB_Dauer_ctrl_np[2,55:77], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[90:130], AVB_AVD_AVB_Dauer_ctrl_np[3,90:130], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[57:80], AVB_AVD_AVB_Dauer_ctrl_np[4,57:80], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[123:150], AVB_AVD_AVB_Dauer_ctrl_np[5,123:150], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[25:58], AVB_AVD_AVB_Dauer_ctrl_np[6,25:58], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[63:93], AVB_AVD_AVB_Dauer_ctrl_np[7,63:93], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[70:82], AVB_AVD_AVB_Dauer_ctrl_np[8,70:82], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[75:95], AVB_AVD_AVB_Dauer_ctrl_np[9,75:95], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[73:97], AVB_AVD_AVB_Dauer_ctrl_np[10,73:97], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[94:135], AVB_AVD_AVB_Dauer_ctrl_np[11,94:135], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[98:140], AVB_AVD_AVB_Dauer_ctrl_np[12,98:140], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[60:80], AVB_AVD_AVB_Dauer_ctrl_np[13,60:80], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[127:147], AVB_AVD_AVB_Dauer_ctrl_np[14,127:147], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[77:83], AVB_AVD_AVB_Dauer_ctrl_np[15,77:83], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[97:130], AVB_AVD_AVB_Dauer_ctrl_np[16,97:130], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[62:102], AVB_AVD_AVB_Dauer_ctrl_np[17,62:102], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[65:98], AVB_AVD_AVB_Dauer_ctrl_np[18,65:98], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[110:140], AVB_AVD_AVB_Dauer_ctrl_np[19,110:140], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[52:62], AVB_AVD_AVB_Dauer_ctrl_np[20,52:62], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[108:143], AVB_AVD_AVB_Dauer_ctrl_np[21,108:143], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[40:75], AVB_AVD_AVB_Dauer_ctrl_np[22,40:75], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[32:76], AVB_AVD_AVB_Dauer_ctrl_np[23,32:76], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[150:177], AVB_AVD_AVB_Dauer_ctrl_np[24,150:177], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[60:103], AVB_AVD_AVB_Dauer_ctrl_np[25,60:103], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[73:99], AVB_AVD_AVB_Dauer_ctrl_np[26,73:99], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[93:150], AVB_AVD_AVB_Dauer_ctrl_np[27,93:150], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[94:130], AVB_AVD_AVB_Dauer_ctrl_np[28,94:130], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[48:80], AVB_AVD_AVB_Dauer_ctrl_np[29,48:80], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[50:72], AVB_AVD_AVB_Dauer_ctrl_np[30,50:72], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[51:69], AVB_AVD_AVB_Dauer_ctrl_np[31,51:69], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[68:82], AVB_AVD_AVB_Dauer_ctrl_np[32,68:82], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[60:82], AVB_AVD_AVB_Dauer_ctrl_np[33,60:82], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[71:101], AVB_AVD_AVB_Dauer_ctrl_np[34,71:101], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[82:102], AVB_AVD_AVB_Dauer_ctrl_np[35,82:102], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[70:82], AVB_AVD_AVB_Dauer_ctrl_np[36,70:82], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[70:83], AVB_AVD_AVB_Dauer_ctrl_np[37,70:83], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[45:70], AVB_AVD_AVB_Dauer_ctrl_np[38,45:70], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[62:77], AVB_AVD_AVB_Dauer_ctrl_np[39,62:77], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[46:80], AVB_AVD_AVB_Dauer_ctrl_np[40,46:80], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[45:80], AVB_AVD_AVB_Dauer_ctrl_np[41,45:80], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[48:72], AVB_AVD_AVB_Dauer_ctrl_np[42,48:72], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[45:75], AVB_AVD_AVB_Dauer_ctrl_np[43,45:75], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])
BDB_Dauer_ctrl_popt, BDB_Dauer_ctrl_pcov = curve_fit(one_phase_decay, signal_x[72:83], AVB_AVD_AVB_Dauer_ctrl_np[44,72:83], method="lm"); AVB_AVD_AVB_Dauer_ctrl_onep.append(BDB_Dauer_ctrl_popt[1])

print("AVB Dauer ctrl: ", len(AVB_AVD_AVB_Dauer_ctrl_onep), "\n", AVB_AVD_AVB_Dauer_ctrl_onep)


In [ ]:
print("AVB; Adult ctrl vs. Adult cx36 : ", stats.ranksums(AVB_AVD_AVB_Adult_ctrl_onep, AVB_AVD_AVB_cx36_onep)[1])
print("AVB; Adult ctrl vs. Dauer ctrl : ", stats.ranksums(AVB_AVD_AVB_Adult_ctrl_onep, AVB_AVD_AVB_Dauer_ctrl_onep)[1])
print("AVB; Dauer ctrl vs. Adult cx36 : ", stats.ranksums(AVB_AVD_AVB_Dauer_ctrl_onep, AVB_AVD_AVB_cx36_onep)[1])

In [ ]:
# plot (box) :: manuscript ver.

    # svg ; change color ; change font


b_width = 0.5; v_width = 0.6; ssz=14; fsz=14


fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(3.5,4))

ax1.set_title("AVB", x=0.5, y=1.01, fontsize=18, fontname='sans-serif')
ax1.set_ylabel("Time (s)", fontsize=14, fontname='sans-serif')
xpos = [1,2,3]

ax1.boxplot(AVB_AVD_AVB_Adult_ctrl_onep, positions=[xpos[0]], widths=b_width, patch_artist=True, whis=(5,95),\
            showmeans=False, showfliers=False, medianprops={"color": "k", "linewidth": 1.0},\
            boxprops={"facecolor": ('#1F77B4'), "edgecolor": "k", "linewidth": 1.6, "alpha": 0.6},\
            whiskerprops={"color": "k", "linewidth": 1.6}, capprops={"color": "k", "linewidth": 1.6})
ax1.boxplot(AVB_AVD_AVB_Dauer_ctrl_onep, positions=[xpos[2]], widths=b_width, patch_artist=True, whis=(5,95),\
            showmeans=False, showfliers=False, medianprops={"color": "k", "linewidth": 1.0},\
            boxprops={"facecolor": ('#FF7F0E'), "edgecolor": "k", "linewidth": 1.6, "alpha": 0.6},\
            whiskerprops={"color": "k", "linewidth": 1.6}, capprops={"color": "k", "linewidth": 1.6})
ax1.boxplot(AVB_AVD_AVB_cx36_onep, positions=[xpos[1]], widths=b_width, patch_artist=True, whis=(5,95),\
            showmeans=False, showfliers=False, medianprops={"color": "k", "linewidth": 1.0},\
            boxprops={"facecolor": ('#8C564B'), "edgecolor": "k", "linewidth": 1.6, "alpha": 0.6},\
            whiskerprops={"color": "k", "linewidth": 1.6}, capprops={"color": "k", "linewidth": 1.6})

ax1.set_xlim(0.3,3.7)
ax1.set_ylim(-0.12,4.50)
label = ["Cx36 X\nAdult","Cx36 O\nAdult","\nDauer"]
ax1.set_xticks(xpos)
ax1.set_xticklabels(label, fontsize=fsz, rotation=0, fontname='sans-serif')

ax1.plot([0.9,2.1], [3.7,3.7], lw=0.8, c='k')
ax1.text(1.5, 3.8, "*", ha='center', va='bottom', color='k', fontsize=14, fontname='sans-serif')
ax1.plot([1.9,3.1], [3.0,3.0], lw=0.8, c='k')
ax1.text(2.5, 3.1, "n.s", ha='center', va='bottom', color='k', fontsize=14, fontname='sans-serif')
fig.tight_layout(pad=0.4)

plt.show()


### AVB=AVD ; AVD

#### Adult with Cx36

In [ ]:
# normalize

AVB_AVD_AVD_Adult_comp_np_norm = np.zeros(AVB_AVD_AVD_Adult_comp_np.shape)


for ii in range(AVB_AVD_AVD_Adult_comp_np_norm.shape[0]):
    
    SIGNAL = np.copy(AVB_AVD_AVD_Adult_comp_np[ii,:])
    MIN = np.min(SIGNAL); MAX = np.max(SIGNAL)
    
    AVB_AVD_AVD_Adult_comp_np_norm[ii,:] = (SIGNAL - MIN) / (MAX - MIN)


In [ ]:
# normalized ver.
# fitting - AVD, Adult with Cx36

    # check X range

AVB_AVD_AVD_cx36_onep_norm = []

BDD_cx36_popt, BDD_cx36_pcov = curve_fit(one_phase_rise, signal_x[30:65], AVB_AVD_AVD_Adult_comp_np_norm[0,30:65], method="lm"); AVB_AVD_AVD_cx36_onep_norm.append(BDD_cx36_popt[1])
BDD_cx36_popt, BDD_cx36_pcov = curve_fit(one_phase_rise, signal_x[14:67], AVB_AVD_AVD_Adult_comp_np_norm[1,14:67], method="lm"); AVB_AVD_AVD_cx36_onep_norm.append(BDD_cx36_popt[1])
BDD_cx36_popt, BDD_cx36_pcov = curve_fit(one_phase_rise, signal_x[20:75], AVB_AVD_AVD_Adult_comp_np_norm[2,20:75], method="lm"); AVB_AVD_AVD_cx36_onep_norm.append(BDD_cx36_popt[1])
BDD_cx36_popt, BDD_cx36_pcov = curve_fit(one_phase_rise, signal_x[27:90], AVB_AVD_AVD_Adult_comp_np_norm[3,27:90], method="lm"); AVB_AVD_AVD_cx36_onep_norm.append(BDD_cx36_popt[1])
BDD_cx36_popt, BDD_cx36_pcov = curve_fit(one_phase_rise, signal_x[30:90], AVB_AVD_AVD_Adult_comp_np_norm[4,30:90], method="lm"); AVB_AVD_AVD_cx36_onep_norm.append(BDD_cx36_popt[1])
BDD_cx36_popt, BDD_cx36_pcov = curve_fit(one_phase_rise, signal_x[20:52], AVB_AVD_AVD_Adult_comp_np_norm[5,20:52], method="lm"); AVB_AVD_AVD_cx36_onep_norm.append(BDD_cx36_popt[1])
BDD_cx36_popt, BDD_cx36_pcov = curve_fit(one_phase_rise, signal_x[10:55], AVB_AVD_AVD_Adult_comp_np_norm[6,10:55], method="lm"); AVB_AVD_AVD_cx36_onep_norm.append(BDD_cx36_popt[1])
BDD_cx36_popt, BDD_cx36_pcov = curve_fit(one_phase_rise, signal_x[10:70], AVB_AVD_AVD_Adult_comp_np_norm[7,10:70], method="lm"); AVB_AVD_AVD_cx36_onep_norm.append(BDD_cx36_popt[1])
BDD_cx36_popt, BDD_cx36_pcov = curve_fit(one_phase_rise, signal_x[0:75], AVB_AVD_AVD_Adult_comp_np_norm[8,0:75], method="lm"); AVB_AVD_AVD_cx36_onep_norm.append(BDD_cx36_popt[1])
BDD_cx36_popt, BDD_cx36_pcov = curve_fit(one_phase_rise, signal_x[25:68], AVB_AVD_AVD_Adult_comp_np_norm[9,25:68], method="lm"); AVB_AVD_AVD_cx36_onep_norm.append(BDD_cx36_popt[1])
BDD_cx36_popt, BDD_cx36_pcov = curve_fit(one_phase_rise, signal_x[5:95], AVB_AVD_AVD_Adult_comp_np_norm[10,5:95], method="lm"); AVB_AVD_AVD_cx36_onep_norm.append(BDD_cx36_popt[1])
BDD_cx36_popt, BDD_cx36_pcov = curve_fit(one_phase_rise, signal_x[0:100], AVB_AVD_AVD_Adult_comp_np_norm[11,0:100], method="lm"); AVB_AVD_AVD_cx36_onep_norm.append(BDD_cx36_popt[1])

print("AVD Adult Cx36: ", len(AVB_AVD_AVD_cx36_onep_norm), "\n", AVB_AVD_AVD_cx36_onep_norm)


In [ ]:
print("AVD; Adult ctrl vs. Adult cx36 : ", stats.ranksums(AVB_AVD_AVD_Adult_ctrl_onep_norm, AVB_AVD_AVD_cx36_onep_norm)[1])
print("AVD; Adult ctrl vs. Dauer ctrl : ", stats.ranksums(AVB_AVD_AVD_Adult_ctrl_onep_norm, AVB_AVD_AVD_Dauer_ctrl_onep_norm)[1])
print("AVD; Dauer ctrl vs. Adult cx36 : ", stats.ranksums(AVB_AVD_AVD_Dauer_ctrl_onep_norm, AVB_AVD_AVD_cx36_onep_norm)[1])

In [ ]:
# plot

b_width = 0.5; v_width = 0.6; ssz=14; fsz=14


fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(3.5,4))

ax1.set_title("AVD", x=0.5, y=1.01, fontsize=18, fontname='sans-serif')
ax1.set_ylabel("Time (s)", fontsize=14, fontname='sans-serif')
xpos = [1,2,3]

ax1.boxplot(ASH_AVD_AVD_Adult_ctrl_onep_norm, positions=[xpos[0]], widths=b_width, patch_artist=True, whis=(5,95),\
            showmeans=False, showfliers=False, medianprops={"color": "k", "linewidth": 1.0},\
            boxprops={"facecolor": ('#1F77B4'), "edgecolor": "k", "linewidth": 1.6, "alpha": 0.6},\
            whiskerprops={"color": "k", "linewidth": 1.6}, capprops={"color": "k", "linewidth": 1.6})
ax1.boxplot(ASH_AVD_AVD_Dauer_ctrl_onep_norm, positions=[xpos[2]], widths=b_width, patch_artist=True, whis=(5,95),\
            showmeans=False, showfliers=False, medianprops={"color": "k", "linewidth": 1.0},\
            boxprops={"facecolor": ('#FF7F0E'), "edgecolor": "k", "linewidth": 1.6, "alpha": 0.6},\
            whiskerprops={"color": "k", "linewidth": 1.6}, capprops={"color": "k", "linewidth": 1.6})
ax1.boxplot(AVB_AVD_AVD_cx36_onep_norm, positions=[xpos[1]], widths=b_width, patch_artist=True, whis=(5,95),\
            showmeans=False, showfliers=False, medianprops={"color": "k", "linewidth": 1.0},\
            boxprops={"facecolor": ('#8C564B'), "edgecolor": "k", "linewidth": 1.6, "alpha": 0.6},\
            whiskerprops={"color": "k", "linewidth": 1.6}, capprops={"color": "k", "linewidth": 1.6})

ax1.set_xlim(0.3,3.7)
ax1.set_ylim(-0.09,8.99)
label = ["Cx36 X\nAdult","Cx36 O\nAdult","\nDauer"]
ax1.set_xticks(xpos)
ax1.set_xticklabels(label, fontsize=fsz, rotation=0, fontname='sans-serif')

ax1.plot([0.9,2.1], [5.0,5.0], lw=0.8, c='k')
ax1.text(1.5, 5.1, "***", ha='center', va='bottom', color='k', fontsize=14, fontname='sans-serif')
ax1.plot([1.9,3.1], [3.3,3.3], lw=0.8, c='k')
ax1.text(2.5, 3.5, "n.s", ha='center', va='bottom', color='k', fontsize=14, fontname='sans-serif')
fig.tight_layout(pad=0.4)

plt.show()


In [ ]:
# get mean, std, sem (normalized ver.)

from scipy.stats import sem

AVB_AVD_AVD_Adult_comp_norm_mean = np.nanmean(AVB_AVD_AVD_Adult_comp_np_norm, axis=0)
AVB_AVD_AVD_Adult_comp_norm_std = np.nanstd(AVB_AVD_AVD_Adult_comp_np_norm, axis=0)
AVB_AVD_AVD_Adult_comp_norm_sem = sem(AVB_AVD_AVD_Adult_comp_np_norm, nan_policy='omit', axis=0)

AVB_AVD_AVD_Adult_ctrl_norm_mean = np.nanmean(AVB_AVD_AVD_Adult_ctrl_np_norm, axis=0)
AVB_AVD_AVD_Adult_ctrl_norm_std = np.nanstd(AVB_AVD_AVD_Adult_ctrl_np_norm, axis=0)
AVB_AVD_AVD_Adult_ctrl_norm_sem = sem(AVB_AVD_AVD_Adult_ctrl_np_norm, nan_policy='omit', axis=0)

AVB_AVD_AVD_Dauer_ctrl_norm_mean = np.nanmean(AVB_AVD_AVD_Dauer_ctrl_np_norm, axis=0)
AVB_AVD_AVD_Dauer_ctrl_norm_std = np.nanstd(AVB_AVD_AVD_Dauer_ctrl_np_norm, axis=0)
AVB_AVD_AVD_Dauer_ctrl_norm_sem = sem(AVB_AVD_AVD_Dauer_ctrl_np_norm, nan_policy='omit', axis=0)


In [ ]:
# plot

fig, ((ax1)) = plt.subplots(1, 1, sharex=True, sharey=True, figsize=(5,4))

xpos = np.arange(1,201,1)
ax1.set_title("AVD", y=1.01, fontsize=18, fontname='sans-serif')
ax1.set_xlabel("Time (s)", fontsize=14, fontname='sans-serif')
ax1.set_ylabel("Fnorm", fontsize=14, fontname='sans-serif')

ax1.plot(xpos, AVB_AVD_AVD_Adult_ctrl_norm_mean, color='#1F77B4', alpha=0.6, linestyle='dashed', linewidth=2.0, label="Adult")
ax1.plot(xpos, AVB_AVD_AVD_Dauer_ctrl_norm_mean, color='#FF7F0E', alpha=0.6, linestyle='dashed', linewidth=2.0, label="Dauer")
ax1.plot(xpos, AVB_AVD_AVD_Adult_comp_norm_mean, color='#8C564B', alpha=0.6, linestyle='dashed', linewidth=2.0, label="Adult + AVB=AVD Cx36")

ax1.fill_between(xpos, AVB_AVD_AVD_Adult_ctrl_norm_mean-AVB_AVD_AVD_Adult_ctrl_norm_sem/2, AVB_AVD_AVD_Adult_ctrl_norm_mean+AVB_AVD_AVD_Adult_ctrl_norm_sem/2,\
                 alpha=0.3, edgecolor=None, facecolor='#1F77B4', linewidth=0, linestyle='dashdot')
ax1.fill_between(xpos, AVB_AVD_AVD_Dauer_ctrl_norm_mean-AVB_AVD_AVD_Dauer_ctrl_norm_sem/2, AVB_AVD_AVD_Dauer_ctrl_norm_mean+AVB_AVD_AVD_Dauer_ctrl_norm_sem/2,\
                 alpha=0.3, edgecolor=None, facecolor='#FF7F0E', linewidth=0, linestyle='dashdot')
ax1.fill_between(xpos, AVB_AVD_AVD_Adult_comp_norm_mean-AVB_AVD_AVD_Adult_comp_norm_sem/2, AVB_AVD_AVD_Adult_comp_norm_mean+AVB_AVD_AVD_Adult_comp_norm_sem/2,\
                 alpha=0.3, edgecolor=None, facecolor='#8C564B', linewidth=0, linestyle='dashdot')

ax1.axvspan(50, 100, alpha=0.15, color='tab:red')

ax1.legend(bbox_to_anchor=(1.02, 1.0), loc='upper left', fontsize=14, ncol=1)
ax1.set_xlim(-5,205)
ax1.set_ylim(0.14,0.899)
ax1.set_xticks(np.arange(0,225,50))
ax1.set_xticklabels(["0","5","10","15","20"], fontsize=11.5, fontname='sans-serif', rotation=0)

plt.show()


---
---